In [51]:
import yaml
from Bio import SeqIO, Entrez
import os
import pandas as pd
from io import StringIO


In [52]:
output_dir = './temp_refseq'
os.makedirs(output_dir, exist_ok=True)

!wget  -P {output_dir} https://ftp.ncbi.nlm.nih.gov/genomes/refseq/bacteria/assembly_summary.txt



7[Files: 0  Bytes: 0  [0 B/s] Re]87[https://ftp.ncbi.nlm.nih.gov/g]87Saving './temp_refseq/assembly_summary.txt'
87assembly_summary.txt   0% [<=>                           ]   71.16K    --.-KB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87assembly_summary.txt   0% [ <=>                          ]  292.54K  221.15KB/s87assembly_summary.txt   0% [  <=>                         ]  687.85K  308.18KB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87assembly_summary.txt   0% [   <=>                        ]    1.38M  447.87KB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87assembly_summary.txt   0% [    <=>                       ]    2.05M  507.84KB/s87assembly_summary.txt   0% [     <=>                      ]    2.50M  499.57KB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87assembly_summary.txt   0% [      <=>                     ]    3.16M  528.31KB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87assembly_summary.txt   0% [       <=>      

In [63]:
def filter_assembly_summary(csv_file_in, csv_file_out):
    info_all_genome_df = pd.read_csv(csv_file_in, sep="\t",low_memory=False, usecols=["assembly_accession", "assembly_level", "ftp_path"])

    df_filtered = info_all_genome_df[
        (info_all_genome_df["assembly_level"] == "Complete Genome") &  # Filtre "Complete Genome"
        (info_all_genome_df["ftp_path"].notna()) &                     # Supprime les "na"
        (info_all_genome_df["ftp_path"].str.startswith("https"))       # Garde uniquement les URL HTTPS
    ].drop(columns=["assembly_level"]).reset_index(drop=True)
    df_filtered.to_csv(csv_file_out, sep="\t", index=False)
    print(f"Filter all (nb {len(info_all_genome_df) })by 'Complete genome' in assembly level (nb {len(df_filtered)})")
    return df_filtered

In [54]:
with open(f"{output_dir}/assembly_summary.txt", "r") as file:
    lines = file.readlines()[1:]  # Supprime la première ligne
    lines[0] = lines[0].lstrip("#")  # Enlève le '#' du début de l'en-tête

df = pd.read_csv(StringIO("".join(lines)), sep="\t", low_memory=False) # Convertir en DataFrame directement
df.to_csv(f"{output_dir}/assembly_summary.csv", sep="\t", index=False) # Sauvegarder en CSV propre

In [64]:

pd.set_option('display.width', 300)  # Augmente la largeur totale de l'affichage
pd.set_option('display.max_colwidth', None)  # Affiche les colonnes complètement

csv_file_in = f"{output_dir}/assembly_summary.csv"
csv_file_out = f"{output_dir}/assembly_summary_filtered_Complete_Genome.csv"
df_filtered_complete_genome = filter_assembly_summary(csv_file_in, csv_file_out)

print(df_filtered_complete_genome)


Filter all (nb 404882)by 'Complete genome' in assembly level (nb 46754)
      assembly_accession                                                                                    ftp_path
0        GCF_900128725.1  https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/900/128/725/GCF_900128725.1_BCifornacula_v1.0
1        GCF_003044255.1        https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/003/044/255/GCF_003044255.1_ASM304425v1
2        GCF_009730575.1        https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/009/730/575/GCF_009730575.1_ASM973057v1
3        GCF_016406305.1       https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/016/406/305/GCF_016406305.1_ASM1640630v1
4        GCF_016406325.1       https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/016/406/325/GCF_016406325.1_ASM1640632v1
...                  ...                                                                                         ...
46749    GCF_046717445.1          https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/046/717/445/GCF_046717445.1_La

In [65]:
assembly_accession,https_path = df_filtered_complete_genome.loc[0]
ftp_path = https_path[8:]
end_url_file = ftp_path.split('/')[-1]
real_ftp_path = f"{ftp_path}/{end_url_file}_genomic.fna.gz"
file_path = f"{output_dir}/{end_url_file}_genomic.fna.gz"
print("ftp url :", real_ftp_path)
os.system(f"wget -P {output_dir} {ftp_path}/{end_url_file}_genomic.fna.gz")
print("download filename ", file_path)
os.system(f"gzip -d {file_path}")
file_path = file_path.removesuffix(".gz")
print("unzipped filename ", file_path)


ftp url : ftp.ncbi.nlm.nih.gov/genomes/all/GCF/900/128/725/GCF_900128725.1_BCifornacula_v1.0/GCF_900128725.1_BCifornacula_v1.0_genomic.fna.gz
HSTS in effect for ftp.ncbi.nlm.nih.gov:80
[0] Downloading 'https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/900/128/725/GCF_900128725.1_BCifornacula_v1.0/GCF_900128725.1_BCifornacula_v1.0_genomic.fna.gz' ...
Saving './temp_refseq/GCF_900128725.1_BCifornacula_v1.0_genomic.fna.gz'
HTTP response 200 OK [https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/900/128/725/GCF_900128725.1_BCifornacula_v1.0/GCF_900128725.1_BCifornacula_v1.0_genomic.fna.gz]
download filename  ./temp_refseq/GCF_900128725.1_BCifornacula_v1.0_genomic.fna.gz
unzipped filename  ./temp_refseq/GCF_900128725.1_BCifornacula_v1.0_genomic.fna


gzip: ./temp_refseq/GCF_900128725.1_BCifornacula_v1.0_genomic.fna already exists;	not overwritten


In [66]:
with open(file_path, "r", encoding='utf-8') as reader:
    first_line = reader.readline()
    print(first_line)
    accession = first_line.split('.')[0][1:]

>NZ_LT667500.1 Buchnera aphidicola strain BCifornacula voucher 2912 chromosome 1



In [67]:
yaml_file = "../import_dataset/refseq/NCBI_credentials.yaml"

with open(yaml_file, 'r') as file:
    credentials = yaml.safe_load(file)

Entrez.email = credentials.get("email")  # Remplacez par votre email
Entrez.api_key = credentials.get("api_key") #""
Entrez.max_tries = 5
Entrez.sleep_between_tries = 15

In [59]:
with Entrez.efetch(db="nucleotide", id=accession, rettype="gb", retmode="text") as taxo_handle:
    x = SeqIO.read(taxo_handle, 'genbank')
    classif = x.annotations['taxonomy']
    sub = x.annotations['organism']

print(classif)

['Bacteria', 'Pseudomonadati', 'Pseudomonadota', 'Gammaproteobacteria', 'Enterobacterales', 'Erwiniaceae', 'Buchnera']


In [60]:
TAXO_LEVELS = ["domain", "phylum", "group", "order", "family", "specie"] 
# From Siegfried
order: int | str = 0
for e in classif:
    if e[-4:] == 'ales':
        order = e
if order:
    group = classif[2] if classif[2][-4:] != 'ales' else classif[1]
    taxo_list = [classif[0], classif[1],  group, order, sub.split(' ')[0], sub.split(' ')[1]]
    print(*list(zip(TAXO_LEVELS, taxo_list)), sep='\n')
    file_name: str = f"{output_dir}/{"_".join(taxo_list)}.fna"
    print(f' change filename to  {file_name}")')
else:
    print(f' rm  {file_path}")')

('domain', 'Bacteria')
('phylum', 'Pseudomonadati')
('group', 'Pseudomonadota')
('order', 'Enterobacterales')
('family', 'Buchnera')
('specie', 'aphidicola')
 change filename to  ./temp_refseq/Bacteria_Pseudomonadati_Pseudomonadota_Enterobacterales_Buchnera_aphidicola.fna")


In [ ]:
# GET TAXO BY BATCH
# accession_map = {}

# with open(decompressed_path, "r", encoding='utf-8') as reader:
#             # first_line = reader.readline().strip()
#             first_line = reader.readline()
#             accession = first_line.split('.')[0][1:]  # Extraction de l'accession
#             accession_map[accession] = decompressed_path
#             decompressed_files.append(decompressed_path)

# accessions = list(accession_map.keys())
# for i in tqdm(range(0, len(accessions), batch_size), desc="Fetching taxonomy data"):
#     batch = accessions[i:i + batch_size]
#     try:1
#         with Entrez.efetch(db="nucleotide", id=batch, rettype="gb", retmode="text") as taxo_handle:
#             # records = SeqIO.read(taxo_handle, 'genbank')
#             records = SeqI%ùO.par